# Mesh Studio trên Google Colab — *Microsoft TRELLIS (1.2B)*
Tạo model 3D chất lượng cao từ 1 ảnh bằng **Microsoft `TRELLIS-image-large`** — sinh đồng thời **Khung 3D + 3D Gaussian 360°** và nướng **UV Texture Map 2048×2048** sắc nét ở cả 4 mặt (Trước, Trái, Phải, Sau).

**Cách dùng:** `Runtime ▸ Change runtime type ▸ T4 GPU`, rồi bấm chạy lần lượt từng ô từ **1 ➔ 8** (Shift+Enter).

> ⚠️ Kết quả (`.glb`, `.obj`, ảnh 4 góc, video xoay 360°) được lưu tự động vào Google Drive (`MyDrive/MeshStudio/data/outputs`).

## 1) Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type > T4 GPU"
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")

## 2) Kết nối Google Drive & chọn nơi lưu
`CACHE_ON_DRIVE = True` lưu model tải về trên Drive để lần sau không phải tải lại.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

CACHE_ON_DRIVE = True
BASE = '/content/drive/MyDrive/MeshStudio'
os.makedirs(BASE, exist_ok=True)

CACHE = f'{BASE}/cache' if CACHE_ON_DRIVE else '/content/cache'
os.environ['HF_HOME'] = f'{CACHE}/huggingface'
os.environ['U2NET_HOME'] = f'{CACHE}/u2net'
os.environ['TORCH_HOME'] = f'{CACHE}/torch'
os.environ['MESH_DATA_DIR'] = f'{BASE}/data'
for p in (CACHE, f'{BASE}/data', f'{CACHE}/u2net'):
    os.makedirs(p, exist_ok=True)
print('Đã thiết lập thư mục lưu tại:', BASE)

## 3) Tải mã nguồn Mesh Studio & Microsoft TRELLIS

In [ ]:
REPO_URL = 'https://github.com/vinhlephu-lpv/MeshP.git'
!rm -rf /content/MeshP
!git clone --depth 1 {REPO_URL} /content/MeshP
!GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://huggingface.co/spaces/JeffreyXiang/TRELLIS /content/TRELLIS 2>/dev/null || git -C /content/TRELLIS pull -q
print('Đã tải xong mã nguồn Mesh Studio & Microsoft TRELLIS!')

## 4) Cài đặt môi trường CUDA cho Microsoft TRELLIS (~1.5 phút)
Dùng `uv` tạo môi trường Python 3.10 độc lập để nạp các bản wheel CUDA biên dịch sẵn (`spconv`, `xformers`, `nvdiffrast`, `diff_gaussian_rasterization`) mà không phải chờ biên dịch C++.

In [ ]:
!pip install -q uv
!uv venv /content/py310 --python 3.10 -q
!uv pip install --python /content/py310/bin/python -q \
    --extra-index-url https://download.pytorch.org/whl/cu121 \
    torch==2.4.0 torchvision==0.19.0 xformers==0.0.27.post2 \
    spconv-cu120==2.3.6 ninja pillow==10.4.0 imageio==2.36.1 imageio-ffmpeg==0.5.1 \
    tqdm easydict opencv-python-headless scipy rembg onnxruntime trimesh \
    xatlas pyvista pymeshfix igraph transformers==4.46.3 huggingface-hub hf_transfer \
    "git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8" \
    "https://huggingface.co/spaces/JeffreyXiang/TRELLIS/resolve/main/wheels/diff_gaussian_rasterization-0.0.0-cp310-cp310-linux_x86_64.whl" \
    "https://huggingface.co/spaces/JeffreyXiang/TRELLIS/resolve/main/wheels/nvdiffrast-0.3.3-cp310-cp310-linux_x86_64.whl"
print('✅ Đã cài xong môi trường Microsoft TRELLIS!')

## 5) Cài đặt thông số chất lượng
- `TEXTURE_SIZE = 2048`: Độ phân giải UV Texture 360° (có thể đổi thành `1024` nếu muốn xuất nhanh hơn).
- `SIMPLIFY = 0.92`: Tỉ lệ rút gọn lưới để model nhẹ, mượt khi đưa vào Unity / Blender.

In [ ]:
TEXTURE_SIZE = 2048
SIMPLIFY = 0.92
SEED = 42

## 6) Tải ảnh lên & Chạy Microsoft TRELLIS
Chọn 1 hoặc nhiều ảnh (nền sẽ được tự động xoá). Sau khi chọn ảnh xong, chạy tiếp ô bên dưới để AI dựng 3D + nướng UV Texture 360°.

In [ ]:
from google.colab import files
import os, shutil
shutil.rmtree('/content/inputs', ignore_errors=True)
os.makedirs('/content/inputs', exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    open(f'/content/inputs/{name}', 'wb').write(data)
print('Đã nhận:', list(uploaded))

In [ ]:
!git -C /content/MeshP pull -q
!/content/py310/bin/python /content/MeshP/backend/mesh_core/trellis_colab.py \
    --input-dir /content/inputs \
    --texture-size {TEXTURE_SIZE} \
    --simplify {SIMPLIFY} \
    --seed {SEED}

## 7) Xem kết quả 4 góc nhìn 360° & Video xoay

In [ ]:
import glob, json, os
from base64 import b64encode
from IPython.display import display, HTML, Image as IPImage

out_root = os.environ.get('MESH_DATA_DIR', '/content/drive/MyDrive/MeshStudio/data') + '/outputs'
metas = sorted(glob.glob(f'{out_root}/*/meta.json'))
results = [json.load(open(p, encoding='utf-8')) for p in metas[-3:]]

if not results:
    print('⚠️ Chưa có model nào được tạo. Hãy chạy ô 6 trước nhé!')
else:
    for m in results:
        print('===', m['name'], '|', m['stats'], '===')
        turn_path = m['dir'] + '/turnaround.png'
        if os.path.exists(turn_path):
            display(IPImage(filename=turn_path))
        vid_path = m['dir'] + '/preview_360.mp4'
        if os.path.exists(vid_path):
            mp4 = open(vid_path, 'rb').read()
            b64 = b64encode(mp4).decode()
            display(HTML(f'<video width="640" controls autoplay loop muted><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

## 8) Đóng gói & Tải về máy (`.glb`, `.obj`, Texture 2K, Video 360°)
Toàn bộ kết quả cũng đã được lưu an toàn trong Google Drive (`MyDrive/MeshStudio/data/outputs`).

In [ ]:
import shutil, os, glob, json
from google.colab import files

out_root = os.environ.get('MESH_DATA_DIR', '/content/drive/MyDrive/MeshStudio/data') + '/outputs'
metas = sorted(glob.glob(f'{out_root}/*/meta.json'))
results = [json.load(open(p, encoding='utf-8')) for p in metas[-3:]]

zips = []
for m in results:
    z = shutil.make_archive(f"/content/{m['slug']}_trellis", 'zip', m['dir'])
    zips.append(z)
    print('Đã nén:', z)
if zips:
    files.download(zips[-1])
    print('Toàn bộ kết quả trong Drive:', out_root)
else:
    print('⚠️ Chưa có model nào để tải.')